# 콘텐츠 기반 필터링 따라가기

팀원 전처리(`preprocess_content_based.py`)와 같은 방식으로,
영화 장르 벡터 → 유저 프로필 → 코사인 유사도 → 1~5점 예측까지
한 셀씩 실행하며 확인합니다.

나중에 `src/content_based.py`의 `fit` / `predict` / `recommend`에 그대로 옮기면 됩니다.

**데이터 위치:** 팀원이 올려 둔 `preprocessing_and_data/movielens-1m/`
(원본 `ratings.dat`, `movies.dat`는 git에 올리지 않는 것이 원칙이지만, 이 폴더에 이미 있습니다.)


## 0. 라이브러리와 경로

노트북을 `notebooks/`에서 열어도, 프로젝트 루트에서 열어도 동작하게 합니다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

ROOT = Path.cwd()
if (ROOT / "src").exists() is False:
    ROOT = ROOT.parent

RAW_DIR = ROOT / "preprocessing_and_data" / "movielens-1m"
print("프로젝트 루트:", ROOT)
print("원본 데이터:", RAW_DIR)
print("movies.dat 있음:", (RAW_DIR / "movies.dat").exists())
print("ratings.dat 있음:", (RAW_DIR / "ratings.dat").exists())


프로젝트 루트: c:\Users\MYCOM\Desktop\학교\4-1\머신러닝\project\machine_learning_term_project
원본 데이터: c:\Users\MYCOM\Desktop\학교\4-1\머신러닝\project\machine_learning_term_project\preprocessing_and_data\movielens-1m
movies.dat 있음: True
ratings.dat 있음: True


## 1. 영화 메타데이터 읽기

MovieLens는 `::`로 칸을 나눕니다. 제목에 특수문자가 있어서 `ISO-8859-1`(latin-1)로 읽습니다.

팀원 코드는 컬럼명이 `movie_id`입니다. 프로젝트 공통 API는 `item_id`이므로
읽은 뒤에 이름을 바꿉니다.


In [2]:
movies = pd.read_csv(
    RAW_DIR / "movies.dat",
    sep="::",
    engine="python",
    encoding="ISO-8859-1",
    names=["item_id", "title", "genres"],
)
print(movies.shape)
movies.head()


(3883, 3)


,item_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


한 영화에 장르가 여러 개 붙어 있습니다. `|`로 나눠 리스트로 만듭니다.


In [3]:
print("예시 행:")
print(movies.iloc[0].to_dict())
print()
print("장르를 나눈 결과:", movies.iloc[0]["genres"].split("|"))
print("고유 장르 미리보기:", sorted({g for row in movies["genres"] for g in row.split("|")})[:10], "...")


예시 행:
{'item_id': 1, 'title': 'Toy Story (1995)', 'genres': "Animation|Children's|Comedy"}

장르를 나눈 결과: ['Animation', "Children's", 'Comedy']
고유 장르 미리보기: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir'] ...


## 2. 장르 멀티핫 인코딩

각 영화를 장르 개수만큼의 0/1 벡터로 만듭니다.
예: Action, Comedy면 Action=1, Comedy=1, 나머지=0.

`MultiLabelBinarizer`가 이 변환을 합니다.


In [4]:
genre_lists = movies["genres"].str.split("|").tolist()
mlb = MultiLabelBinarizer()
movie_multihot = mlb.fit_transform(genre_lists)

movie_feature_df = pd.DataFrame(
    movie_multihot,
    index=movies["item_id"],
    columns=mlb.classes_,
)
print("장르 목록:", list(mlb.classes_))
print("행렬 크기 (영화 수, 장르 수):", movie_feature_df.shape)
movie_feature_df.head()


장르 목록: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
행렬 크기 (영화 수, 장르 수): (3883, 18)


,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
item_id,,,,,,,,,,,,,,,,,,
1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


첫 영화의 벡터를 장르 이름과 같이 보면, 1인 칸이 그 영화의 장르입니다.


In [5]:
first_id = movies.iloc[0]["item_id"]
row = movie_feature_df.loc[first_id]
print("영화:", movies.iloc[0]["title"])
print("장르 문자열:", movies.iloc[0]["genres"])
print()
print(row[row == 1])


영화: Toy Story (1995)
장르 문자열: Animation|Children's|Comedy

Animation     1
Children's    1
Comedy        1
Name: 1, dtype: int64


## 3. 영화 벡터 L2 정규화

길이가 다른 벡터를 그대로 내적하면 장르가 많은 영화가 유리합니다.
각 벡터를 길이 1로 나누면, 이후 내적이 곧 **코사인 유사도**가 됩니다.

팀원 코드와 동일합니다.


In [6]:
movie_norms = np.linalg.norm(movie_multihot, axis=1, keepdims=True)
movie_norms[movie_norms == 0] = 1.0
movie_multihot_norm = movie_multihot / movie_norms

print("정규화 전 첫 벡터 길이:", float(np.linalg.norm(movie_multihot[0])))
print("정규화 후 첫 벡터 길이:", float(np.linalg.norm(movie_multihot_norm[0])))

item_index = {int(item_id): i for i, item_id in enumerate(movies["item_id"])}
item_ids = movies["item_id"].to_numpy()
titles = movies["title"].to_numpy()


정규화 전 첫 벡터 길이: 1.7320508075688772
정규화 후 첫 벡터 길이: 1.0


## 4. 평점 읽기

공통 컬럼 `user_id`, `item_id`, `rating`으로 맞춥니다.


In [7]:
ratings = pd.read_csv(
    RAW_DIR / "ratings.dat",
    sep="::",
    engine="python",
    encoding="ISO-8859-1",
    names=["user_id", "item_id", "rating", "timestamp"],
)
print(ratings.shape)
print(ratings["rating"].value_counts().sort_index())
ratings.head()


(1000209, 4)
rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64


,user_id,item_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


## 5. 학습/테스트 분할 (누수 방지)

팀원이 저장한 `user_profiles_data.npz`는 **전체 평점**으로 유저 프로필을 만듭니다.
테스트 평점까지 들어가면 평가가 낙관적으로 나옵니다.

여기서는 item-based와 같이 **사용자별 80/20 분할**을 하고,
프로필은 **학습 평점만**으로 만듭니다.


In [8]:
def split_by_user(ratings, test_size=0.2, random_state=42):
    """사용자마다 일부를 테스트로 빼고, 학습은 최소 1개를 남긴다."""
    frame = ratings.reset_index(drop=True)
    rng = np.random.default_rng(random_state)
    mask = np.zeros(len(frame), dtype=bool)
    for positions in frame.groupby("user_id", sort=False).indices.values():
        if len(positions) > 1:
            count = min(len(positions) - 1, max(1, int(len(positions) * test_size)))
            mask[rng.choice(positions, count, replace=False)] = True
    return frame.loc[~mask].copy(), frame.loc[mask].copy()


train, test = split_by_user(ratings)
print("학습:", len(train), "테스트:", len(test))
print("학습에만 있는 사용자 수:", train["user_id"].nunique())


학습: 802553 테스트: 197656
학습에만 있는 사용자 수: 6040


## 6. 유저 프로필 만들기 (학습 평점만)

팀원과 같은 식입니다.

1. `adjusted_rating = rating - 3`  (3점은 선호/비선호 경계)
2. 그 영화의 장르 벡터에 이 값을 곱함  
   → 좋아하는 장르는 +, 싫어하는 장르는 −
3. 사용자별로 더함
4. L2 정규화

4점을 준 Action 영화: Action 칸에 +1  
2점을 준 Horror 영화: Horror 칸에 −1


In [9]:
train = train.copy()
train["adjusted_rating"] = train["rating"] - 3.0

merged = train.merge(movie_feature_df, left_on="item_id", right_index=True)
genre_cols = list(mlb.classes_)
for col in genre_cols:
    merged[col] = merged[col] * merged["adjusted_rating"]

user_profiles_raw = merged.groupby("user_id")[genre_cols].sum()
raw_values = user_profiles_raw.to_numpy(dtype=float)
norms = np.linalg.norm(raw_values, axis=1, keepdims=True)
norms[norms == 0] = 1.0
user_profiles_norm = raw_values / norms

user_ids = user_profiles_raw.index.to_numpy()
user_index = {int(u): i for i, u in enumerate(user_ids)}
print("유저 프로필 크기:", user_profiles_norm.shape)
user_profiles_raw.head()


유저 프로필 크기: (6040, 18)


,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
user_id,,,,,,,,,,,,,,,,,,
1,6.0,5.0,13.0,18.0,14.0,2.0,0.0,26.0,3.0,0.0,0.0,15.0,0.0,4.0,4.0,2.0,4.0,0.0
2,19.0,11.0,0.0,0.0,13.0,7.0,0.0,53.0,0.0,0.0,0.0,0.0,1.0,13.0,5.0,7.0,9.0,4.0
3,19.0,22.0,3.0,3.0,20.0,0.0,0.0,7.0,3.0,0.0,-1.0,1.0,0.0,2.0,5.0,4.0,2.0,10.0
4,19.0,0.0,0.0,1.0,0.0,2.0,0.0,7.0,1.0,0.0,3.0,0.0,0.0,2.0,5.0,4.0,1.0,3.0
5,-10.0,0.0,4.0,4.0,20.0,7.0,3.0,-2.0,0.0,3.0,-2.0,2.0,2.0,-1.0,1.0,-5.0,3.0,1.0


한 명의 프로필을 장르 이름과 같이 봅니다. 값이 큰 장르를 상대적으로 좋아한 것입니다.


In [10]:
example_user = int(user_ids[0])
profile = pd.Series(user_profiles_raw.loc[example_user], index=genre_cols)
print("user_id:", example_user)
print(profile.sort_values(ascending=False))
print()
print("이 사용자가 학습에서 준 평점 일부:")
train.loc[train["user_id"] == example_user, ["item_id", "rating"]].head()


user_id: 1
Drama          26.0
Children's     18.0
Musical        15.0
Comedy         14.0
Animation      13.0
Action          6.0
Adventure       5.0
Sci-Fi          4.0
Romance         4.0
War             4.0
Fantasy         3.0
Thriller        2.0
Crime           2.0
Documentary     0.0
Film-Noir       0.0
Horror          0.0
Mystery         0.0
Western         0.0
Name: 1, dtype: float64

이 사용자가 학습에서 준 평점 일부:


,item_id,rating
0,1193,5
1,661,3
2,914,3
5,1197,3
6,1287,5


## 7. 예측: 코사인 → 1~5점

유저 벡터와 영화 벡터가 둘 다 길이 1이면

`유사도 = np.dot(user_vec, item_vec)`  ∈ [−1, 1]

전처리가 3점 기준이므로 평점으로 이렇게 바꿉니다.

`점수 = clip(3 + 2 × 유사도, 1, 5)`

- 유사도 1 → 5점
- 유사도 0 → 3점
- 유사도 −1 → 1점

모르는 사용자/영화는 학습 평균으로 대체합니다 (item-based와 같은 생각).


In [11]:
global_mean = float(train["rating"].mean())
user_means = train.groupby("user_id")["rating"].mean().to_dict()
item_means = train.groupby("item_id")["rating"].mean().to_dict()
rating_min = float(train["rating"].min())
rating_max = float(train["rating"].max())


def predict(user_id, item_id):
    u = user_index.get(int(user_id))
    i = item_index.get(int(item_id))
    if u is not None and i is not None:
        sim = float(np.dot(user_profiles_norm[u], movie_multihot_norm[i]))
        score = 3.0 + 2.0 * sim
        return float(np.clip(score, rating_min, rating_max)), "content", 1
    if user_id in user_means:
        return float(user_means[user_id]), "user_mean", 0
    if item_id in item_means:
        return float(item_means[item_id]), "item_mean", 0
    return global_mean, "global_mean", 0


uid = 1
sample_item = int(test.loc[test["user_id"] == uid, "item_id"].iloc[0])
pred, method, n = predict(uid, sample_item)
actual = float(test.loc[(test["user_id"] == uid) & (test["item_id"] == sample_item), "rating"].iloc[0])
title = titles[item_index[sample_item]]
print(f"user={uid}, item={sample_item} ({title})")
print(f"실제 평점: {actual}, 예측: {pred:.3f}, method={method}")


user=1, item=3408 (Erin Brockovich (2000))
실제 평점: 4.0, 예측: 4.255, method=content


## 8. 추천: 안 본 영화 중 점수 상위 K

학습에서 이미 평가한 아이템은 빼고,
점수는 내림차순, 동점은 `item_id` 오름차순입니다.
반환 컬럼은 item-based와 같습니다.


In [12]:
def recommend(user_id, top_n=10):
    seen = set(train.loc[train["user_id"] == user_id, "item_id"])
    records = []
    for item_id in item_ids:
        if int(item_id) in seen:
            continue
        score, method, n = predict(user_id, int(item_id))
        records.append((int(item_id), score, method, n))
    rec = pd.DataFrame(
        records,
        columns=["item_id", "predicted_rating", "method", "n_contributors"],
    )
    rec = rec.sort_values(["predicted_rating", "item_id"], ascending=[False, True])
    rec = rec.head(top_n).reset_index(drop=True)
    rec["title"] = rec["item_id"].map(lambda i: titles[item_index[i]])
    return rec


recommend(1, top_n=10)


,item_id,predicted_rating,method,n_contributors,title
0,34,4.616735,content,1,Babe (1995)
1,1014,4.616735,content,1,Pollyanna (1960)
2,1812,4.616735,content,1,Wide Awake (1998)
3,1288,4.533110,content,1,This Is Spinal Tap (1984)
4,241,4.502135,content,1,Fluke (1995)
5,262,4.502135,content,1,"Little Princess, A (1995)"
6,755,4.502135,content,1,Kim (1950)
7,917,4.502135,content,1,"Little Princess, The (1939)"
8,1012,4.502135,content,1,Old Yeller (1957)
9,1013,4.502135,content,1,"Parent Trap, The (1961)"


## 9. (선택) 작은 평가

테스트 전체를 돌리면 시간이 꽤 걸립니다.
앞에서 2000개만 MAE/RMSE를 봅니다.

전체 평가는 나중에 `scripts/run_content_based.py`로 돌리면 됩니다.


In [13]:
sample_test = test.head(2000)
preds = [predict(u, i)[0] for u, i in sample_test[["user_id", "item_id"]].itertuples(index=False, name=None)]
err = np.asarray(preds) - sample_test["rating"].to_numpy()
print("n =", len(sample_test))
print("MAE =", float(np.abs(err).mean()))
print("RMSE =", float(np.sqrt(np.mean(err ** 2))))


n = 2000
MAE = 0.8741410700539225
RMSE = 1.1050941721048717


## 다음에 할 일

이 노트북에서 확인한 함수를 `src/content_based.py`에 옮기면 됩니다.

- `fit(train_ratings)`: 영화 벡터 로드(또는 생성) + **학습만으로** 유저 프로필
- `predict(user_id, item_id) -> float`
- `recommend(user_id, top_n=10) -> DataFrame`

팀원 `.npz`를 쓸 때:

- `movies_multihot_data.npz`의 영화 벡터는 그대로 써도 됩니다 (장르는 평점과 무관).
- `user_profiles_data.npz`는 전체 평점으로 만들어져 있으니, 공정한 평가에는 `fit`에서 다시 계산하세요.
